In [1]:
import os

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType
from pyspark.sql.window import Window

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, Imputer

BASE_DIR = "/app"
DATA_RAW = os.path.join(BASE_DIR, "data", "raw")
DATA_INTERIM = os.path.join(BASE_DIR, "data", "interim")
DATA_PROCESSED = os.path.join(BASE_DIR, "data", "processed")
OUTPUTS_DIR = os.path.join(BASE_DIR, "outputs")

spark = (
    SparkSession.builder
    .appName("02_Feature_Engineering_Person2")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8") # So it doesn't go crazy with shuffles
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark ready | version:", spark.version)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/12 19:40:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/02/12 19:40:06 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark ready | version: 4.1.1


In [2]:
import os

# If running in Colab with upload, keep this as fallback.
# In Docker/Jupyter, we use the file produced by Notebook 01 in data/interim.
csv_file = os.path.join(DATA_INTERIM, "valuation_cleaned_updated.csv")

if not os.path.exists(csv_file):
    raise FileNotFoundError(
        f"Expected cleaned file not found at: {csv_file}\n"
        "Run notebooks/01_cleaning.ipynb first (it should write valuation_cleaned_updated.csv)."
    )
else:
    print("Found:", csv_file)

Found: /app/data/interim/valuation_cleaned_updated.csv


In [3]:
import glob

# Being a bit forgiving with file naming (CSV vs csv)
# In Docker/Jupyter we read from the structured path set in Cell 1.
print("Using file:", csv_file)

df_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("nullValue", "null")  # treats literal "null" strings as real nulls
    .csv(csv_file)
)

print(f"Loaded  | rows={df_raw.count()} cols={len(df_raw.columns)}")
df_raw.printSchema()

df_raw.show(5, truncate=False)

Using file: /app/data/interim/valuation_cleaned_updated.csv


Loaded  | rows=88092 cols=15
root
 |-- procedure_name_en: string (nullable = true)
 |-- procedure_year: integer (nullable = true)
 |-- procedure_number: integer (nullable = true)
 |-- instance_date: timestamp (nullable = true)
 |-- actual_worth: double (nullable = true)
 |-- procedure_area: double (nullable = true)
 |-- property_type_id: integer (nullable = true)
 |-- property_type_en: string (nullable = true)
 |-- property_sub_type_id: integer (nullable = true)
 |-- property_sub_type_en: string (nullable = true)
 |-- area_id: integer (nullable = true)
 |-- area_name_en: string (nullable = true)
 |-- actual_area: double (nullable = true)
 |-- property_total_value: double (nullable = true)
 |-- actual_area_missing: boolean (nullable = true)

+-------------------+--------------+----------------+-------------------+------------+--------------+----------------+----------------+--------------------+--------------------+-------+----------------+-----------+--------------------+--------------

In [4]:
# We want a stable row_id (1..N) for reproducible joins.
# Fix: break ties deterministically if multiple rows share the same hash.

base_cols = sorted(df_raw.columns)  # snapshot BEFORE adding IDs

row_hash_expr = F.sha2(
    F.concat_ws("||", *[F.coalesce(F.col(c).cast("string"), F.lit("NULL")) for c in base_cols]),
    256
)

df_raw = df_raw.withColumn("row_hash", row_hash_expr)

# tie-breakers: add the full row content as secondary sort keys (string-cast)
tie_breakers = [F.coalesce(F.col(c).cast("string"), F.lit("NULL")).alias(c) for c in base_cols]

w = Window.orderBy(F.col("row_hash"), *[F.col(c) for c in base_cols])
df_raw = df_raw.withColumn("row_id", F.row_number().over(w)).drop("row_hash")

print("Added row_id (deterministic even with duplicates)")
rows = df_raw.count()
distinct_ids = df_raw.select("row_id").distinct().count()
print("rows:", rows, "| distinct row_id:", distinct_ids)
assert rows == distinct_ids, "row_id is not unique (unexpected)"
df_raw.agg(F.min("row_id").alias("min_id"), F.max("row_id").alias("max_id")).show()


Added row_id (deterministic even with duplicates)


26/02/12 19:40:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/02/12 19:40:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/02/12 19:40:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/02/12 19:40:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/02/12 19:40:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/02/12 19:40:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


rows: 88092 | distinct row_id: 88092


26/02/12 19:40:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/02/12 19:40:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+------+------+
|min_id|max_id|
+------+------+
|     1| 88092|
+------+------+



In [5]:
# Dropping columns that usually don't help ML, or are duplicates of the EN name columns.
# We do it only if they exist so the notebook doesn't crash.

redundant = [
    "procedure_name_en",
    "procedure_number",
    "instance_date",
    "property_type_id",
    "property_sub_type_id",
    "area_id"
]

to_drop = [c for c in redundant if c in df_raw.columns]
df = df_raw.drop(*to_drop)

print("Dropped:", to_drop)
print("Remaining cols:", len(df.columns))


Dropped: ['procedure_name_en', 'procedure_number', 'instance_date', 'property_type_id', 'property_sub_type_id', 'area_id']
Remaining cols: 10


In [6]:
# Minimal type cleanup (not re-doing Person 1 cleaning, just making sure types are usable)

numeric_cols = ["procedure_year", "actual_worth", "procedure_area", "actual_area", "property_total_value"]
for c in numeric_cols:
    if c in df.columns:
        df = df.withColumn(c, F.col(c).cast(DoubleType()))

# Sometimes this flag comes as "true"/"false" text in CSV
if "actual_area_missing" in df.columns:
    df = df.withColumn(
        "actual_area_missing",
        F.when(F.col("actual_area_missing") == "true", F.lit(1)).otherwise(F.lit(0)).cast(IntegerType())
    )
else:
    # If not provided, we can derive it from actual_area being null
    if "actual_area" in df.columns:
        df = df.withColumn("actual_area_missing", F.col("actual_area").isNull().cast(IntegerType()))

# If target source missing -> can't label it
before = df.count()
df = df.dropna(subset=["property_total_value"])
after = df.count()

print(f"Dropped {before - after} rows where property_total_value was null")
print("Rows now:", after)


Dropped 0 rows where property_total_value was null
Rows now: 88092


In [7]:
#  procedure_area == 0 looks like invalid/missing.
# We DON'T overwrite procedure_area (so Person 1 cleaned file meaning stays intact).
# Instead:
#  procedure_area_was_zero: tells the model a repair happened
# procedure_area_fixed: usable numeric feature (uses actual_area if available)

if "procedure_area" not in df.columns or "actual_area" not in df.columns:
    raise ValueError("Expected procedure_area and actual_area columns in the cleaned dataset.")

df = df.withColumn("procedure_area_was_zero", (F.col("procedure_area") == 0.0).cast(IntegerType()))

df = df.withColumn(
    "procedure_area_fixed",
    F.when((F.col("procedure_area") == 0.0) & (F.col("actual_area") > 0), F.col("actual_area"))
     .when(F.col("procedure_area") == 0.0, F.lit(None).cast(DoubleType()))
     .otherwise(F.col("procedure_area"))
).withColumn("procedure_area_fixed", F.col("procedure_area_fixed").cast(DoubleType()))

# quick stats so you can mention it in your report
zero_rows = df.filter(F.col("procedure_area") == 0.0).count()
fixed_nulls = df.filter(F.col("procedure_area_fixed").isNull()).count()
print("procedure_area==0 rows:", zero_rows)
print("procedure_area_fixed nulls (will be imputed):", fixed_nulls)

df.filter(F.col("procedure_area") == 0.0).select(
    "row_id", "procedure_area", "actual_area", "procedure_area_fixed", "procedure_area_was_zero"
).show(5, truncate=False)


procedure_area==0 rows: 841
procedure_area_fixed nulls (will be imputed): 1


26/02/12 19:40:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/02/12 19:40:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/02/12 19:40:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/02/12 19:40:25 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/02/12 19:40:25 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+------+--------------+-----------+--------------------+-----------------------+
|row_id|procedure_area|actual_area|procedure_area_fixed|procedure_area_was_zero|
+------+--------------+-----------+--------------------+-----------------------+
|21    |0.0           |450.01     |450.01              |1                      |
|36    |0.0           |450.02     |450.02              |1                      |
|230   |0.0           |405.01     |405.01              |1                      |
|284   |0.0           |456.2      |456.2               |1                      |
|337   |0.0           |400.17     |400.17              |1                      |
+------+--------------+-----------+--------------------+-----------------------+
only showing top 5 rows


In [8]:
from pyspark.sql import functions as F

df.select(
    F.count(F.when(F.col("procedure_area") == 0, True)).alias("procedure_area_zero_rows"),
    F.count(F.when((F.col("procedure_area") == 0) & F.col("actual_area").isNull(), True)).alias("zero_and_actual_null"),
    F.count(F.when((F.col("procedure_area") == 0) & F.col("actual_area").isNotNull(), True)).alias("zero_and_actual_present"),
).show()

df.filter(F.col("procedure_area") == 0).select(
    "procedure_area", "actual_area", "procedure_area_fixed", "procedure_area_was_zero"
).show(20, truncate=False)


+------------------------+--------------------+-----------------------+
|procedure_area_zero_rows|zero_and_actual_null|zero_and_actual_present|
+------------------------+--------------------+-----------------------+
|                     841|                   0|                    841|
+------------------------+--------------------+-----------------------+

+--------------+-----------+--------------------+-----------------------+
|procedure_area|actual_area|procedure_area_fixed|procedure_area_was_zero|
+--------------+-----------+--------------------+-----------------------+
|0.0           |450.01     |450.01              |1                      |
|0.0           |857.45     |857.45              |1                      |
|0.0           |450.01     |450.01              |1                      |
|0.0           |405.0      |405.0               |1                      |
|0.0           |505.97     |505.97              |1                      |
|0.0           |613.48     |613.48             

In [9]:
# Just making categoricals consistent (no re-cleaning beyond that).
# Empty strings are a pain in Spark; better to turn them into "Unknown".

cat_cols = [c for c in ["property_type_en", "property_sub_type_en", "area_name_en"] if c in df.columns]
print("Categorical cols:", cat_cols)

for c in cat_cols:
    df = df.withColumn(c, F.trim(F.col(c)))
    df = df.withColumn(
        c,
        F.when((F.col(c).isNull()) | (F.col(c) == ""), F.lit("Unknown")).otherwise(F.col(c))
    )

df.select(cat_cols).show(11, truncate=False)


Categorical cols: ['property_type_en', 'property_sub_type_en', 'area_name_en']
+----------------+--------------------+------------------+
|property_type_en|property_sub_type_en|area_name_en      |
+----------------+--------------------+------------------+
|Land            |Commercial          |Hor Al Anz        |
|Land            |Commercial          |Al Goze Fourth    |
|Land            |Residential         |Al Manara         |
|Land            |Commercial          |Abu Hail          |
|Land            |Commercial          |Al Suq Al Kabeer  |
|Land            |Commercial          |Al Safouh Second  |
|Land            |Commercial          |Hor Al Anz East   |
|Land            |Commercial          |AL QUSAIS         |
|Land            |Residential         |Al Khawaneej First|
|Land            |Commercial          |Port Saeed        |
|Land            |Parking             |Trade Center First|
+----------------+--------------------+------------------+
only showing top 11 rows


In [10]:
# Balanced bins using quantiles so classes aren't super imbalanced
cuts = df.approxQuantile("property_total_value", [1/3, 2/3], 0.001)
low_cut, high_cut = cuts[0], cuts[1]

print(f"Cutoffs -> Low <= {low_cut:,.0f}, Medium <= {high_cut:,.0f}, High > {high_cut:,.0f}")

df = df.withColumn(
    "valuation_class",
    F.when(F.col("property_total_value") <= F.lit(low_cut), F.lit("Low"))
     .when(F.col("property_total_value") <= F.lit(high_cut), F.lit("Medium"))
     .otherwise(F.lit("High"))
)

# Deterministic label mapping (no StringIndexer randomness)
df = df.withColumn(
    "label",
    F.when(F.col("valuation_class") == "Low", F.lit(0.0))
     .when(F.col("valuation_class") == "Medium", F.lit(1.0))
     .otherwise(F.lit(2.0))
).withColumn("label", F.col("label").cast(DoubleType()))

df.groupBy("valuation_class", "label").count().orderBy("label").show()


Cutoffs -> Low <= 2,600,000, Medium <= 12,642,698, High > 12,642,698
+---------------+-----+-----+
|valuation_class|label|count|
+---------------+-----+-----+
|            Low|  0.0|29332|
|         Medium|  1.0|29324|
|           High|  2.0|29436|
+---------------+-----+-----+



In [11]:
# Pipeline order matters:
# 1)Impute numeric cols (median) so vectors don't contain nulls
# 2) Index categoricals (keep invalid so CV folds don't crash)
# 3) One-hot encode categoricals, also keep invalid
# 4)Assemble everything into features and strict mode: no silent row dropping)

stages = []

# numeric imputation (only for cols we actually use)
impute_inputs = []
impute_inputs = []
for c in ["procedure_year", "procedure_area_fixed", "actual_area"]:
    if c in df.columns:
        impute_inputs.append(c)

impute_outputs = [c + "_imputed" for c in impute_inputs]

if impute_inputs:
    imputer = Imputer(inputCols=impute_inputs, outputCols=impute_outputs).setStrategy("median")
    stages.append(imputer)

# categorical encoding
index_cols = []
ohe_cols = []

for c in cat_cols:
    idx = c + "_idx"
    ohe = c + "_ohe"
    stages.append(StringIndexer(inputCol=c, outputCol=idx, handleInvalid="keep"))
    index_cols.append(idx)
    ohe_cols.append(ohe)

# multi-col OneHotEncoder (cleaner), with fallback if handleInvalid may not be supported
if index_cols:
    try:
        enc = OneHotEncoder(inputCols=index_cols, outputCols=ohe_cols, handleInvalid="keep")
    except TypeError:
        enc = OneHotEncoder(inputCols=index_cols, outputCols=ohe_cols)
    stages.append(enc)

# numeric features to feed into VectorAssembler
candidate_nums = []

# keep procedure_year if present
if "procedure_year_imputed" in impute_outputs:
    candidate_nums.append("procedure_year_imputed")
elif "procedure_year" in df.columns:
    candidate_nums.append("procedure_year")

# use imputed numerics (engineered)
for c in ["procedure_area_fixed_imputed", "actual_area_imputed"]:
    if c in impute_outputs:
        candidate_nums.append(c)

# flags (good signals for models)
for c in ["actual_area_missing", "procedure_area_was_zero"]:
    if c in df.columns:
        candidate_nums.append(c)

feature_cols = candidate_nums + ohe_cols

print("Feature inputs going into assembler:")
for i, c in enumerate(feature_cols, 1):
    print(f"  {i}. {c}")

# Strict assembler: error if something is invalid (no silent dropping)
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features", handleInvalid="error")
stages.append(assembler)

pipeline = Pipeline(stages=stages)
print("Pipeline built  | stages:", len(stages))


Feature inputs going into assembler:
  1. procedure_year_imputed
  2. procedure_area_fixed_imputed
  3. actual_area_imputed
  4. actual_area_missing
  5. procedure_area_was_zero
  6. property_type_en_ohe
  7. property_sub_type_en_ohe
  8. area_name_en_ohe
Pipeline built  | stages: 6


In [12]:
# Fit/transform once here just to generate the final ML-ready dataset.
#note for team:  member handling cv can still refit per fold later if they want to be extra strict.

df.cache()
before = df.count()

pipe_model = pipeline.fit(df)
df_final = pipe_model.transform(df)

after = df_final.count()
print("Rows before:", before, "| after:", after)
assert before == after, "Row count changed (something got dropped/failed)."

# Making  sure we don't accidentally put IDs into features
assert "row_id" not in feature_cols, "row_id accidentally included in features!"

# ML-ready output ie safe to train on
df_ml = df_final.select("row_id", "features", "label")

# Reference output (note for reporting only, not training)
df_ref = df_final.select("row_id", "valuation_class", "property_total_value")

df_ml.show(5, truncate=False)
df_ref.show(5, truncate=False)


26/02/12 19:40:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/02/12 19:40:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/02/12 19:40:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/02/12 19:40:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/02/12 19:40:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
                                                                                

Rows before: 88092 | after: 88092
+------+---------------------------------------------------------------+-----+
|row_id|features                                                       |label|
+------+---------------------------------------------------------------+-----+
|1     |(291,[0,1,2,5,13,174],[2012.0,172615.17,172615.17,1.0,1.0,1.0])|2.0  |
|2     |(291,[0,1,2,6,11,73],[2012.0,145.86,145.86,1.0,1.0,1.0])       |0.0  |
|3     |(291,[0,1,2,5,12,176],[2017.0,2557.34,2557.34,1.0,1.0,1.0])    |2.0  |
|4     |(291,[0,1,2,5,9,112],[2003.0,1111.86,1111.83,1.0,1.0,1.0])     |2.0  |
|5     |(291,[0,1,2,5,10,109],[2010.0,7569.55,13604.62,1.0,1.0,1.0])   |1.0  |
+------+---------------------------------------------------------------+-----+
only showing top 5 rows
+------+---------------+--------------------+
|row_id|valuation_class|property_total_value|
+------+---------------+--------------------+
|1     |High           |1.486412E8          |
|2     |Low            |1300000.0           |
|

In [13]:
from pyspark.ml.functions import vector_to_array

# basic null checks
null_features = df_ml.filter(F.col("features").isNull()).count()
null_labels = df_ml.filter(F.col("label").isNull()).count()
print("Null features:", null_features, "| Null labels:", null_labels)
assert null_features == 0, "Found null features!"
assert null_labels == 0, "Found null labels!"

# label distribution sanity
print("\nLabel distribution:")
df_ml.groupBy("label").count().orderBy("label").show()

# check NaNs inside vector (these can or may silently destroy training)
nan_rows = (
    df_ml.select(vector_to_array("features").alias("fa"))
    .select(F.expr("exists(fa, x -> isnan(x))").alias("has_nan"))
    .filter("has_nan")
    .count()
)
print("Rows with NaN inside features:", nan_rows)
assert nan_rows == 0, "Found NaNs in feature vectors!"

# feature vector dimension; useful to debug
dim_arr = df_ml.select(vector_to_array("features").alias("fa")).limit(1).collect()[0]["fa"]
print("Feature vector dimension:", len(dim_arr))


Null features: 0 | Null labels: 0

Label distribution:
+-----+-----+
|label|count|
+-----+-----+
|  0.0|29332|
|  1.0|29324|
|  2.0|29436|
+-----+-----+



Rows with NaN inside features: 0
Feature vector dimension: 291


In [14]:
import os
os.makedirs(DATA_PROCESSED, exist_ok=True)

ml_path = os.path.join(DATA_PROCESSED, "valuation_ml_ready.parquet")
ref_path = os.path.join(DATA_PROCESSED, "valuation_label_ref.parquet")

df_ml.write.mode("overwrite").parquet(ml_path)
df_ref.write.mode("overwrite").parquet(ref_path)

print("Saved")
print(" -", ml_path, "(row_id, features, label)")
print(" -", ref_path, "(row_id, valuation_class, property_total_value)")

# reload test so we know it's not corrupted
test_ml = spark.read.parquet(ml_path)
test_ref = spark.read.parquet(ref_path)

print("\nReloaded  | ML rows:", test_ml.count(), "| REF rows:", test_ref.count())
test_ml.show(3, truncate=False)
test_ref.show(3, truncate=False)

Saved
 - /app/data/processed/valuation_ml_ready.parquet (row_id, features, label)
 - /app/data/processed/valuation_label_ref.parquet (row_id, valuation_class, property_total_value)

Reloaded  | ML rows: 88092 | REF rows: 88092
+------+---------------------------------------------------------------+-----+
|row_id|features                                                       |label|
+------+---------------------------------------------------------------+-----+
|1     |(291,[0,1,2,5,13,174],[2012.0,172615.17,172615.17,1.0,1.0,1.0])|2.0  |
|2     |(291,[0,1,2,6,11,73],[2012.0,145.86,145.86,1.0,1.0,1.0])       |0.0  |
|3     |(291,[0,1,2,5,12,176],[2017.0,2557.34,2557.34,1.0,1.0,1.0])    |2.0  |
+------+---------------------------------------------------------------+-----+
only showing top 3 rows
+------+---------------+--------------------+
|row_id|valuation_class|property_total_value|
+------+---------------+--------------------+
|1     |High           |1.486412E8          |
|2     |Low 

In [15]:
# Post-save verification: reload and verify schema
ml_path = os.path.join(DATA_PROCESSED, "valuation_ml_ready.parquet")
ref_path = os.path.join(DATA_PROCESSED, "valuation_label_ref.parquet")

test_ml = spark.read.parquet(ml_path)
test_ref = spark.read.parquet(ref_path)

print("Reloaded ML rows:", test_ml.count())
print("Reloaded REF rows:", test_ref.count())

print("\nML schema:")
test_ml.printSchema()

print("\nREF schema:")
test_ref.printSchema()

test_ml.show(5, truncate=False)
test_ref.show(5, truncate=False)

Reloaded ML rows: 88092
Reloaded REF rows: 88092

ML schema:
root
 |-- row_id: integer (nullable = true)
 |-- features: vector (nullable = true)
 |-- label: double (nullable = true)


REF schema:
root
 |-- row_id: integer (nullable = true)
 |-- valuation_class: string (nullable = true)
 |-- property_total_value: double (nullable = true)

+------+---------------------------------------------------------------+-----+
|row_id|features                                                       |label|
+------+---------------------------------------------------------------+-----+
|1     |(291,[0,1,2,5,13,174],[2012.0,172615.17,172615.17,1.0,1.0,1.0])|2.0  |
|2     |(291,[0,1,2,6,11,73],[2012.0,145.86,145.86,1.0,1.0,1.0])       |0.0  |
|3     |(291,[0,1,2,5,12,176],[2017.0,2557.34,2557.34,1.0,1.0,1.0])    |2.0  |
|4     |(291,[0,1,2,5,9,112],[2003.0,1111.86,1111.83,1.0,1.0,1.0])     |2.0  |
|5     |(291,[0,1,2,5,10,109],[2010.0,7569.55,13604.62,1.0,1.0,1.0])   |1.0  |
+------+--------------------

In [16]:
from pyspark.sql import functions as F
from pyspark.ml.functions import vector_to_array

# 1) Null checks
null_f = test_ml.filter(F.col("features").isNull()).count()
null_l = test_ml.filter(F.col("label").isNull()).count()
print("Null features:", null_f, "| Null labels:", null_l)
assert null_f == 0 and null_l == 0, "Nulls found in ML output!"

# 2) NaN check inside the vector
nan_rows = (
    test_ml.select(vector_to_array("features").alias("fa"))
    .select(F.expr("exists(fa, x -> isnan(x))").alias("has_nan"))
    .filter("has_nan")
    .count()
)
print("Rows with NaN inside features:", nan_rows)
assert nan_rows == 0, "NaNs found inside feature vectors!"

# 3) Feature dimension check
dim_arr = test_ml.select(vector_to_array("features").alias("fa")).limit(1).collect()[0]["fa"]
print("Feature vector dimension:", len(dim_arr))

# 4) Label distribution check
print("\nLabel distribution:")
test_ml.groupBy("label").count().orderBy("label").show()

Null features: 0 | Null labels: 0


Rows with NaN inside features: 0
Feature vector dimension: 291

Label distribution:
+-----+-----+
|label|count|
+-----+-----+
|  0.0|29332|
|  1.0|29324|
|  2.0|29436|
+-----+-----+



In [17]:
#verification
test_ml.groupBy("label").count().orderBy("label").show()

+-----+-----+
|label|count|
+-----+-----+
|  0.0|29332|
|  1.0|29324|
|  2.0|29436|
+-----+-----+



In [18]:
#row_id verifficatiom
from pyspark.sql import functions as F
test_ml.agg(F.min("row_id").alias("min_id"), F.max("row_id").alias("max_id")).show()


+------+------+
|min_id|max_id|
+------+------+
|     1| 88092|
+------+------+



In [19]:
import os
import shutil

os.makedirs(os.path.join(OUTPUTS_DIR, "artifacts"), exist_ok=True)

zip_path = os.path.join(OUTPUTS_DIR, "artifacts", "02_feature_engineering_outputs.zip")
base_name = zip_path.replace(".zip", "")

# Zip the processed outputs (parquet dirs) so it can be shared/submitted if needed
shutil.make_archive(base_name, "zip", DATA_PROCESSED)

print("Created:", zip_path)

Created: /app/outputs/artifacts/02_feature_engineering_outputs.zip
